# Bank Marketing Classification

---

Author: heleneInsights

---

## Objective

This project applies data cleaning, exploratory data analysis, and machine learning models to predict whether a client will subscribe to a term deposit based on the **Bank Marketing Dataset** from the UCI Machine Learning Repository.

---

## Problem Statement

The goal is to build a classification model that predicts the variable **`y`** (whether a client subscribes to a term deposit) using demographic, financial, and campaign-related features.

# Dataset Description

This dataset contains information from direct phone marketing campaigns run by a Portuguese bank. Each record represents a client contact, often with multiple calls per client. The goal is to predict whether a client will subscribe to a term deposit (“yes” or “no”) based on their characteristics and campaign interactions.

## Dataset Source

In this project, we analyze the **UCI Bank Marketing Dataset**, obtained from the UCI Machine Learning Repository. The dataset contains information about **direct marketing campaigns conducted by a Portuguese bank**, where clients were contacted mainly via phone calls to promote term deposit subscriptions.

It includes features describing **client demographics, financial status, past marketing interactions, and macroeconomic conditions**, which are used to predict whether a client will subscribe to a term deposit.

There are **four dataset versions** available in the source:

* Two full datasets
* Two datasets based on **10% random samples**

### Dataset Versions

* **`bank`**: Original dataset with **16 input features + 1 target variable**
* **`bank-additional`**: Extended version with **20 input features + 1 target variable**

According to the `bank-additional-names.txt` file, the extended dataset includes five additional **social and economic indicators**, which significantly improve predictive performance — even when the `duration` feature is excluded (to avoid leakage).

The authors also note that the smaller 10% samples are provided to allow testing of more computationally expensive algorithms, such as **Support Vector Machines (SVM)**.

### Dataset Choice for This Project

For this analysis, we will use the **`bank-additional` dataset (10% sample version)**. This choice allows:

* Faster experimentation and model training
* Easier comparison with computationally intensive algorithms
* Sufficient data quality for meaningful exploratory analysis and classification modeling

### Source Link

The dataset can be accessed and downloaded from the following link:

[https://archive.ics.uci.edu/dataset/222/bank+marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing)

## Dataset Information

* **Number of observations**: 4,119 client records
* **Number of features**: 20 input variables + 1 target variable
* **Target variable**: `y` (term deposit subscription: yes/no)

## Data Dictionary — Bank Marketing Dataset

| Column         | Type  | Description                           | Example             |
| -------------- | ----- | ------------------------------------- | ------------------- |
| age            | int   | Age of the client                     | 39                  |
| job            | str   | Type of job                           | "admin."            |
| marital        | str   | Marital status                        | "married"           |
| education      | str   | Education level                       | "university.degree" |
| default        | str   | Has credit in default?                | "no"                |
| housing        | str   | Has housing loan?                     | "yes"               |
| loan           | str   | Has personal loan?                    | "no"                |
| contact        | str   | Contact communication type            | "cellular"          |
| month          | str   | Last contact month                    | "may"               |
| day_of_week    | str   | Last contact day                      | "fri"               |
| duration       | int   | Last call duration (seconds)          | 487                 |
| campaign       | int   | Number of contacts during campaign    | 2                   |
| pdays          | int   | Days since last contact (999 = never) | 999                 |
| previous       | int   | Previous campaign contacts            | 0                   |
| poutcome       | str   | Previous campaign outcome             | "nonexistent"       |
| emp.var.rate   | float | Employment variation rate             | -1.8                |
| cons.price.idx | float | Consumer price index                  | 92.893              |
| cons.conf.idx  | float | Consumer confidence index             | -46.2               |
| euribor3m      | float | Euribor 3-month rate                  | 1.313               |
| nr.employed    | float | Number of employees indicator         | 5099.1              |
| y              | str   | Target: subscribed term deposit?      | "no"                |

---

### Feature Description

This dataset comes from a **bank marketing campaign**, where a bank tries to convince clients to open a term deposit. Each feature describes either the client, the contact process, or the economic context.

#### Client Demographics (Who the client is)

- **age**: The client’s age in years. Younger and middle-aged clients are more common, with fewer elderly clients.
- **job**: The client’s occupation (e.g., admin, technician, blue-collar). This helps understand the client’s economic profile.
- **marital**: Marital status (married, single, divorced). It gives social background information about the client.
- **education**: Highest education level achieved. It is a proxy for financial literacy and income potential.

#### Financial Situation (Client’s financial conditions)

* **default**: Whether the client has ever failed to repay credit. Most clients are “no”, but many are marked “unknown” (missing information).
* **housing**: Whether the client has a housing loan (mortgage). This shows long-term financial commitment.
* **loan**: Whether the client has a personal loan. Indicates current debt level.

#### Contact Information (How the bank contacted the client)

* **contact**: Type of communication used (cellular or telephone). Most contacts are done via mobile phones.
* **month**: Month when the last contact happened. Strongly concentrated in May, showing seasonality in campaigns.
* **day_of_week**: Day of the week of last contact. Fairly evenly distributed across weekdays.

#### Campaign Interaction (What happened during marketing)

* **duration**: Length of the last phone call in seconds. Longer calls often indicate higher engagement, but this is also a **data leakage feature** (it is only known after the call ends).
* **campaign**: Number of times the client was contacted during this campaign. Most clients receive only a few calls, but some are contacted many times.
* **pdays**: Days since the client was last contacted in a previous campaign. The value **999 means “never contacted before”**, not a real number.
* **previous**: Number of previous contacts before this campaign. Most clients were never contacted before.
* **poutcome**: Result of the previous campaign (success, failure, or nonexistent). Most clients have “nonexistent”.

#### Economic Context (External environment during contact)

These variables describe the **economic situation in Portugal/Europe at the time**, which affects customer behavior:

* **emp.var.rate**: Employment variation rate. Reflects changes in employment levels over time.
* **cons.price.idx**: Consumer price index. Measures inflation and price changes.
* **cons.conf.idx**: Consumer confidence index. Measures how optimistic or pessimistic consumers are.
* **euribor3m**: Interest rate for European banks (3-month Euribor). Strongly affects loans and savings behavior.
* **nr.employed**: Number of employees in the economy. A general indicator of economic strength.

#### Target Variable (What we predict)

* **y**: Whether the client subscribed to a term deposit:

  * “yes” → subscribed
  * “no” → did not subscribe
* Highly imbalanced: most clients do **not subscribe (~89%)**.

#### Key Global Notes

* Many features have **“unknown” values**, which are hidden missing data.
* The dataset is **highly imbalanced**, especially the target variable.
* Some features are **highly skewed or dominated by one value** (e.g., `pdays = 999`).
* `duration` is very informative but can cause **data leakage** in real prediction systems.

---

# Data Loading and Inspection

## Import Libraries

In [ ]:
# Standard library imports
import math
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from scipy.stats import gaussian_kde

### Dataset Availability and Path Setup

In [5]:
data_dir = Path("../data/raw/")

# List available files (for verification)
available_files = list(data_dir.iterdir())
print("Available files:", available_files)
file_name = "bank-additional.csv"
dataset_path = data_dir / file_name

Available files: [WindowsPath('../data/raw/bank-additional.csv')]


In [6]:
# Validate dataset existence
if not dataset_path.exists():
    raise FileNotFoundError(f"{file_name} not found in {data_dir}")

print("Dataset found at:", dataset_path)

Dataset found at: ..\data\raw\bank-additional.csv


## Load Dataset

According to the dataset documentation, the file is semicolon-separated (`;`). Therefore, it was loaded in Python using:

In [9]:
df = pd.read_csv(dataset_path, sep=";")

This is equivalent to the loading example provided in the original dataset documentation for R.

### Display Configuration

To prevent pandas from truncating large outputs during data exploration, the following display settings were applied:

In [20]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

This allows all rows and columns to be displayed when inspecting DataFrames.

## Dataset Overview

### First and Last Records

In [17]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,30,blue-collar,married,basic.9y,no,yes,no,cellular,may,fri,487,2,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
1,39,services,single,high.school,no,no,no,telephone,may,fri,346,4,999,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no
2,25,services,married,high.school,no,yes,no,telephone,jun,wed,227,1,999,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no
3,38,services,married,basic.9y,no,unknown,unknown,telephone,jun,fri,17,3,999,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no
4,47,admin.,married,university.degree,no,yes,no,cellular,nov,mon,58,1,999,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no


In [18]:
df.tail()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
4114,30,admin.,married,basic.6y,no,yes,yes,cellular,jul,thu,53,1,999,0,nonexistent,1.4,93.918,-42.7,4.958,5228.1,no
4115,39,admin.,married,high.school,no,yes,no,telephone,jul,fri,219,1,999,0,nonexistent,1.4,93.918,-42.7,4.959,5228.1,no
4116,27,student,single,high.school,no,no,no,cellular,may,mon,64,2,999,1,failure,-1.8,92.893,-46.2,1.354,5099.1,no
4117,58,admin.,married,high.school,no,no,no,cellular,aug,fri,528,1,999,0,nonexistent,1.4,93.444,-36.1,4.966,5228.1,no
4118,34,management,single,high.school,no,yes,no,cellular,nov,wed,175,1,999,0,nonexistent,-0.1,93.200,-42.0,4.120,5195.8,no


Missing values are present in several columns as `unknown` and will be addressed in the following sections, according with the follow note available in the file bank-additional-names.txt

"Missing Attribute Values: There are several missing values in some categorical attributes, all coded with the "unknown" label. These missing values can be treated as a possible class label or using deletion or imputation techniques. "

### Verify Structure

In [12]:
print(f"{file_name}")
print(f"Shape: {df.shape}")
print(f"→ {df.shape[0]} rows, {df.shape[1]} columns\n")

bank-additional.csv
Shape: (4119, 21)
→ 4119 rows, 21 columns



---

## Initial Data Inspection

### Inspect Data Types

In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4119 entries, 0 to 4118
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             4119 non-null   int64  
 1   job             4119 non-null   str    
 2   marital         4119 non-null   str    
 3   education       4119 non-null   str    
 4   default         4119 non-null   str    
 5   housing         4119 non-null   str    
 6   loan            4119 non-null   str    
 7   contact         4119 non-null   str    
 8   month           4119 non-null   str    
 9   day_of_week     4119 non-null   str    
 10  duration        4119 non-null   int64  
 11  campaign        4119 non-null   int64  
 12  pdays           4119 non-null   int64  
 13  previous        4119 non-null   int64  
 14  poutcome        4119 non-null   str    
 15  emp.var.rate    4119 non-null   float64
 16  cons.price.idx  4119 non-null   float64
 17  cons.conf.idx   4119 non-null   float64
 18 

The dataset has **4,119 rows** and **21 columns** with no missing values in any variable. It includes a mix of numeric (int64 and float64) and categorical variables (currently stored as `str`). The features cover client demographics, loan and contact history, campaign interaction data, and macroeconomic indicators, with the target variable indicating whether the client subscribed to a term deposit.

In [21]:
df.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'duration',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed',
 'y']

### Descriptive Statistics

In [22]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
age,4119.0,40.113620,10.313362,18.000,32.000,38.000,47.000,88.000
duration,4119.0,256.788055,254.703736,0.000,103.000,181.000,317.000,3643.000
campaign,4119.0,2.537266,2.568159,1.000,1.000,2.000,3.000,35.000
pdays,4119.0,960.422190,191.922786,0.000,999.000,999.000,999.000,999.000
previous,4119.0,0.190337,0.541788,0.000,0.000,0.000,0.000,6.000
emp.var.rate,4119.0,0.084972,1.563114,-3.400,-1.800,1.100,1.400,1.400
cons.price.idx,4119.0,93.579704,0.579349,92.201,93.075,93.749,93.994,94.767
cons.conf.idx,4119.0,-40.499102,4.594578,-50.800,-42.700,-41.800,-36.400,-26.900
euribor3m,4119.0,3.621356,1.733591,0.635,1.334,4.857,4.961,5.045
nr.employed,4119.0,5166.481695,73.667904,4963.600,5099.100,5191.000,5228.100,5228.100


#### **Numeric Variables — Summary & Key Observations**

The dataset shows a mix of client behavior variables, campaign interaction metrics, and macroeconomic indicators.

**Client & Campaign Behavior**

- **Age**: the mean is aprox. 40 years (range 18–88). Slightly right-skewed toward middle-aged clients.
- **Campaign (contacts per client)**: Mostly low (median = 2), but with extreme outliers up to 35 calls → possible inefficiency or hard-to-reach clients.
- **Duration**: Highly variable (0–3643 sec), strongly right-skewed. Many short calls; few very long ones.

**Previous Campaign Info**

- **pdays**: Extremely concentrated at **999 (non-contacted clients)** → dominant placeholder value.
- This is a **data encoding issue**: 999 is not a real numeric distance and should be treated as a separate category or missing indicator.
- **previous**: Mostly 0 (median = 0), meaning most clients were not previously contacted.

**Macroeconomic Variables**

- **emp.var.rate**: ranges from -3.4 to 1.4, indicating different economic periods.
- **cons.price.idx**: stable (low variance).
- **cons.conf.idx**: consistently negative → overall pessimistic consumer sentiment.
- **euribor3m**: bimodal-like behavior (low values around 1.3 and high around ~4.8–5.0), suggesting distinct economic regimes.
- **nr.employed**: tightly clustered (4963–5228), low variability.

**Data Quality & Modeling Concerns**

1. **pdays encoding issue (999)**

   - Not a real value → should be converted to “not previously contacted” flag.

2. **duration leakage risk**

   - Strong predictor but only known after call ends → should be removed for realistic prediction.

3. **Outliers**

   - `campaign` (up to 35) and `duration` (very large max) may distort models if not handled.

4. **Skewed distributions**

   - Many variables (previous, campaign, duration) are highly right-skewed → scaling or transformation may help.

In [23]:
df.describe(include="str").T

,count,unique,top,freq
job,4119,12,admin.,1012
marital,4119,4,married,2509
education,4119,8,university.degree,1264
default,4119,3,no,3315
housing,4119,3,yes,2175
loan,4119,3,no,3349
contact,4119,2,cellular,2652
month,4119,10,may,1378
day_of_week,4119,5,thu,860
poutcome,4119,3,nonexistent,3523


#### **Categorical Variables — Summary & Observations**

The dataset is dominated by a few high-frequency categories across most variables, showing clear imbalance in several features.

**Client Profile**

- **Job**: Mostly *admin.* (1012 cases), indicating a strong concentration in office/administrative roles.
- **Marital status**: Majority are *married* (2509), suggesting a middle-aged, stable customer base.
- **Education**: Most common is *university.degree* (1264), indicating relatively well-educated clients.

**Financial Status**

- **Default**: Almost all clients have *no default* (3315), meaning very low credit risk variation.
- **Housing loan**: Slight majority have *yes* (2175), fairly balanced.
- **Personal loan**: Mostly *no* (3349), indicating fewer personal loans than housing loans.

**Contact & Campaign Context**

- **Contact type**: Strong preference for *cellular* (2652 vs telephone), reflecting modern outreach methods.
- **Month**: Highly concentrated in *May* (1378), suggesting seasonal campaign focus or data collection bias.
- **Day of week**: Most contacts on *Thursday* (860), but distribution is relatively spread across weekdays.
- **Previous outcome (poutcome)**: Dominated by *nonexistent* (3523), meaning most clients had no prior campaign history.

**Target Variable**

- **y (subscription)**: Strong class imbalance:

  - *no*: 3668
  - *yes*: 451 (approx.)

This is a key modeling challenge — the dataset is heavily skewed toward non-subscription.

**Key Data Quality & Modeling Insights**

1. **Severe class imbalance**

   - Target variable is highly skewed → accuracy alone will be misleading.

2. **Dominant categories**

   - Many variables (e.g., `poutcome`, `contact`, `month`) are heavily concentrated in one class → limited variability.

3. **Sparse signal in some features**

   - Variables like `default` and `loan` have very low variation, which may reduce predictive power.

4. **Potential temporal bias**

   - Overrepresentation of certain months (May) may bias model toward seasonal effects.

5. **Encoding needed**

   - All categorical variables should be encoded (one-hot or target encoding depending on model choice).

---

### Missing Values

In [25]:
missing_summary = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_summary.sort_values('Missing Values', ascending=False)

,Missing Values,Percentage (%)
age,0,0.0
job,0,0.0
marital,0,0.0
education,0,0.0
default,0,0.0
housing,0,0.0
loan,0,0.0
contact,0,0.0
month,0,0.0
day_of_week,0,0.0


#### Missing Values Summary & Data Quality Note

At a strict numeric level, the dataset contains **no missing values (0% across all columns)**. However, this is misleading because missing information is encoded as **"unknown"** in several categorical variables.

#### Hidden Missingness (“unknown” values)

According to the dataset documentation, several categorical features use `"unknown"` to represent missing or unavailable information. These should be treated as **implicit missing values**, not valid categories.

#### Key Observations

- The dataset is **technically complete**, but **not truly clean** due to encoded missing values.
- `"unknown"` values introduce:

  - **Noise in categorical distributions**
  - Potential **bias if treated as a real category**
  - Need for explicit handling before modeling

#### Data Quality Implications

1. **Hidden missingness**

   * Must be converted from `"unknown"` → `NaN` before analysis.

2. **Modeling risk**

   * Treating `"unknown"` as a valid category may distort relationships.

3. **Recommended handling strategies**

   - Imputation (most common category or model-based)
   - Separate “unknown” indicator feature
   - Or removal if proportion is small

#### Conclusion

While there are no formal missing values, the presence of `"unknown"` labels means the dataset contains **structural missing data that must be handled explicitly** before feature engineering and modeling.

### Duplicate Records Check

In [26]:
n_duplicated = df.duplicated().sum()

duplicate_pct = (
    n_duplicated / len(df) * 100
)

print(f"Exact duplicated rows before cleaning: {n_duplicated}")
print(f"Percentage of duplicated records: {duplicate_pct:.2f}%")

Exact duplicated rows before cleaning: 0
Percentage of duplicated records: 0.00%


#### Duplicate Records Summary

The dataset contains:

- **0 exact duplicate rows**
- **0.00% duplicated records**

#### Interpretation & Data Quality Insight

- The dataset is **clean in terms of exact duplication**, meaning no repeated client records were found.
- This improves reliability for:

  - statistical summaries
  - model training stability
  - unbiased frequency distributions

#### Important Note

Even though there are **no exact duplicates**, it is still possible to have:

- **near-duplicates** (same client contacted multiple times)
- repeated individuals across different campaigns (not identifiable from exact row matching)

Repeated calls to the same client are expected and represent valid campaign behavior, not duplicate records.

---

## Distribution Analysis

### Numeric

#### Histograms

In [32]:
# Select numeric columns
numeric_columns = df.select_dtypes(include=np.number).columns

# Dashboard layout
n_cols = 5
n_rows = math.ceil(len(numeric_columns) / n_cols)

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[
        f"<span style='font-size:12px'>{col}</span>" for col in numeric_columns
    ],
    vertical_spacing=0.18
)

for i, col in enumerate(numeric_columns):
    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1

    data = df[col].dropna()

    # Histogram
    fig.add_trace(
        go.Histogram(
            x=data,
            name=col,
            nbinsx=30,
            opacity=0.6,
            showlegend=False
        ),
        row=row,
        col=col_pos
    )

    # KDE (distribution curve)
    kde = gaussian_kde(data)
    x_range = np.linspace(data.min(), data.max(), 200)
    kde_values = kde(x_range)

    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=kde_values * len(data) * (data.max() - data.min()) / 30,
            mode='lines',
            name=f"{col} KDE",
            line=dict(width=2)
        ),
        row=row,
        col=col_pos
    )

    # Mean line
    fig.add_vline(
        x=data.mean(),
        line_width=2,
        line_dash="dash",
        line_color="green",
        annotation_text="Mean",
        annotation_position="right",
        row=row,
        col=col_pos
    )

    # Median line
    fig.add_vline(
        x=data.median(),
        line_width=2,
        line_dash="dot",
        line_color="red",
        annotation_text="Median",
        annotation_position="bottom right",
        row=row,
        col=col_pos
    )

# Layout updates
fig.update_layout(
    title="Numeric Feature Distributions with KDE, Mean & Median",
    height=300 * n_rows,
    width=1400,
    showlegend=False,
    template="plotly_white",
    bargap=0.05,
    margin=dict(t=60)  
)

fig.update_annotations(
    font_size=12,
    standoff=525,  # increases distance between subplot title and plot
    yshift=10  # pushes subplot titles upward
)

fig.update_xaxes(title_text="Feature Values")
fig.update_yaxes(title_text="Frequency")

fig.show()

##### Distribution Analysis

##### Histograms — Key Observations

- **age**: Right-skewed distribution with most clients around middle age and a long tail toward older ages.
- **duration**: Highly right-skewed, with most calls being short and a few extremely long calls.
- **campaign**: Strong right skew; most clients were contacted few times, with some extreme outliers.
- **pdays**: Highly concentrated at 999 (no previous contact), with very few lower values.
- **previous**: Discrete and highly skewed, with most values at 0 and a long tail of higher counts.
- **emp.var.rate**: Bimodal distribution, mainly concentrated around ~1.4 and -1.8, indicating distinct economic periods.
- **cons.price.idx**: Relatively stable but slightly multimodal around ~93–94, not normally distributed.
- **cons.conf.idx**: Multimodal distribution with peaks around -40 and a spread toward higher values.
- **euribor3m**: Clearly non-normal, with clustering around low (~1–2) and high (~4–5) values.
- **nr.employed**: Strong clustering near 5220 with limited variation and a left-side tail toward lower values.

##### General Insights

- Most variables are **not normally distributed**.
- Several features show **heavy skewness and outliers**, especially:

  - `duration`
  - `campaign`
  - `previous`
  
- Some variables (especially `pdays`) contain **structural values (999)** that dominate the distribution.
- Macroeconomic variables show **clustered regimes rather than smooth distributions**, reflecting real economic cycles.

##### Key Takeaway

This dataset is **highly non-Gaussian and skewed**, so models assuming normality (like linear models without transformation) may need:

- scaling
- log transformations (for skewed features)
- or tree-based models which handle distributions better naturally

### Categorical

#### Categorical Value Consistency

In [33]:
categorical_cols = df.select_dtypes(
    include=["str", "category", "object"]
).columns

for col in categorical_cols:
    print(f"\n{'='*40}")
    print(f"Column: {col}")
    print(f"{'='*40}")

    summary = (
        df[col]
        .value_counts(dropna=False)
        .reset_index()
    )

    summary.columns = [col, 'Count']

    summary['Percentage'] = (
        summary['Count'] / len(df) * 100
    ).round(2)

    print(summary)


Column: job
              job  Count  Percentage
0          admin.   1012       24.57
1     blue-collar    884       21.46
2      technician    691       16.78
3        services    393        9.54
4      management    324        7.87
5         retired    166        4.03
6   self-employed    159        3.86
7    entrepreneur    148        3.59
8      unemployed    111        2.69
9       housemaid    110        2.67
10        student     82        1.99
11        unknown     39        0.95

Column: marital
    marital  Count  Percentage
0   married   2509       60.91
1    single   1153       27.99
2  divorced    446       10.83
3   unknown     11        0.27

Column: education
             education  Count  Percentage
0    university.degree   1264       30.69
1          high.school    921       22.36
2             basic.9y    574       13.94
3  professional.course    535       12.99
4             basic.4y    429       10.42
5             basic.6y    228        5.54
6              unknow

In [37]:
MAX_CATEGORIES = 10

n_cols = 4
n_rows = math.ceil(len(categorical_cols) / n_cols)

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=categorical_cols,
    vertical_spacing=0.10
)

for i, col in enumerate(categorical_cols):

    summary = (
        df[col]
        .value_counts(dropna=False)
        .head(MAX_CATEGORIES)
        .reset_index()
    )

    summary.columns = [col, "Count"]

    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1

    fig.add_trace(
        go.Bar(
            x=summary[col].astype(str),
            y=summary["Count"],
            text=summary["Count"],
            textposition="outside",
            name=col
        ),
        row=row,
        col=col_pos
    )

fig.update_layout(
    title="Categorical Variables Distribution Dashboard",
    height=400 * n_rows,
    width=1400,
    showlegend=False,
    template="plotly_white"
)

fig.show()

##### Categorical Variables — Distribution Analysis

##### Job

- The dataset is dominated by **admin (24.6%)**, followed by **blue-collar (21.5%)** and **technician (16.8%)**.
- Overall, the sample is concentrated in middle- to low-income occupational groups.
- Rare categories like *student* and *unknown* are under 3%, which may limit their predictive power.

##### Marital Status

- Strong majority are **married (60.9%)**, followed by **single (28%)**.
- Very small proportion of **unknown (0.3%)**, so minimal impact from missing categorization.

##### Education

- Most clients have **university degree (30.7%)** or **high school (22.4%)**.
- Basic education levels are also common, but **illiterate cases are almost nonexistent (0.02%)**.
- Around **4% unknown**, which should be handled carefully.

##### Credit Status

- **Default**: overwhelmingly **no (80.5%)**, with a large portion marked as **unknown (19.5%)**.
- **Housing loan**: relatively balanced (yes 52.8%, no 44.7%).
- **Personal loan**: mostly **no (81.3%)**, with a small unknown share (2.6%).

##### Contact Information

- Most clients were contacted via **cellular (64.4%)**, suggesting modern outreach dominates.

##### Temporal Variables

- **Month**: Highly concentrated in **May (33.5%)**, followed by July and August → strong seasonality.
- **Day of week**: Very evenly distributed (~18–21% each), indicating no strong weekday bias.

##### Previous Campaign Outcome

- Dominated by **nonexistent (85.5%)**, meaning most clients had no prior campaign exposure.
- **Success cases are rare (3.5%)**, indicating strong class imbalance in historical performance.

##### Target Variable (y)

- Highly imbalanced:

  - **No: 89.0%**
  - **Yes: 11.0%**

This is a key modeling challenge and will strongly affect evaluation strategy.

##### Key Insights & Data Quality Considerations

- **Strong class imbalance** in target (`y`) → accuracy is misleading; we will use F1-score, ROC-AUC, or PR-AUC.
- Several features are **heavily skewed toward one category**:

  - `poutcome`, `marital`, `job`

- **Rare categories exist** (e.g., illiterate, unknown values) → may need grouping or encoding strategy.
- **Seasonality is evident in month**, which may introduce temporal bias.
- Presence of **“unknown” values in multiple variables** should be treated as missing data, not valid categories.

##### Overall Interpretation

The dataset reflects a **real-world banking marketing scenario with strong imbalance, categorical concentration, and behavioral skew**, which makes it suitable for:

- classification modeling
- imbalance-aware learning techniques
- feature engineering focused on categorical grouping and encoding

### Countplots

##### IQR calculation

In [40]:
feature_columns = df.select_dtypes(
    include=np.number
).columns

outlier_summary = []

for col in feature_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    n_outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()

    outlier_summary.append({
        "Feature": col,
        "Outliers": n_outliers,
        "Percentage": round(n_outliers / len(df) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df.sort_values(by='Percentage', ascending=False)

,Feature,Outliers,Percentage
4,previous,596,14.47
1,duration,291,7.06
2,campaign,235,5.71
3,pdays,160,3.88
7,cons.conf.idx,43,1.04
0,age,39,0.95
5,emp.var.rate,0,0.00
6,cons.price.idx,0,0.00
8,euribor3m,0,0.00
9,nr.employed,0,0.00


#### Outlier Detection

In [39]:
# Select numeric columns
numeric_columns = df.select_dtypes(
    include=np.number
).columns

if len(numeric_columns) > 0:

    # create boxplots
    # Dashboard layout
    n_cols = 5
    n_rows = math.ceil(len(numeric_columns) / n_cols)

    # Create subplot figure
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=numeric_columns
    )

    # Add boxplots
    for i, col in enumerate(numeric_columns):
        row = (i // n_cols) + 1
        col_pos = (i % n_cols) + 1

        fig.add_trace(
            go.Box(
                y=df[col],
                name=col,
                boxmean=True
            ),
            row=row,
            col=col_pos
        )

    # Update layout
    fig.update_layout(
        title="Interactive Boxplot Dashboard — Numeric Features",
        height=300 * n_rows,
        width=1400,
        showlegend=False,
        template="plotly_white"
    )

    # Add axis labels
    fig.update_yaxes(title_text="Observed Values")

    # Show dashboard
    fig.show()

else:
    print("No numeric variables available for outlier analysis.")

##### Outlier Detection — Summary & Interpretation

##### Key Findings

* **previous (14.47%)**

  * Highest proportion of outliers.
  * This variable is highly skewed because most clients have 0 previous contacts, while a small group has multiple interactions.

* **duration (7.06%)**

  * Strong outlier presence due to a few extremely long calls.
  * Distribution is heavily right-skewed, which is expected in call data.

* **campaign (5.71%)**

  * Some clients were contacted unusually many times (up to 35 calls), creating noticeable outliers.

* **pdays (3.88%)**

  * Outliers mostly come from rare values different from the dominant 999 encoding.
  * Important to interpret carefully because 999 is a special category, not a true numeric value.

* **cons.conf.idx (1.04%)**

  * Minor outliers due to small variations in consumer confidence during economic fluctuations.

* **age (0.95%)**

  * Very few extreme age values (older clients close to 80–90 years).

##### Features with No Outliers Detected

* `emp.var.rate`
* `cons.price.idx`
* `euribor3m`
* `nr.employed`

These macroeconomic indicators are relatively stable and well-behaved, with no extreme deviations detected.

##### Overall Insights

* Outliers are mainly concentrated in **behavioral and interaction variables**, not economic ones.
* Most outliers are **not errors**, but rather reflect real-world behavior (e.g., repeated calls, long conversations).
* However, some variables (especially `campaign`, `duration`, `previous`) may need:

  * robust scaling (e.g., median/IQR-based methods)
  * or transformation (log scale) depending on the model

##### Key Takeaway

The dataset contains **meaningful but highly skewed behavioral outliers**, which should be handled carefully rather than blindly removed, as they may carry predictive signal.

---

# Exploratory Data Analysis (EDA)

### Correlation Analysis

The correlation matrix reveals the strength and direction of the linear relationships between the numerical variables.

In [41]:
numeric_columns = df.select_dtypes(
    include=np.number
).columns

corr_matrix = df[numeric_columns].corr()

corr_matrix

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
age,1.000000,0.041299,-0.014169,-0.043425,0.050931,-0.019192,-0.000482,0.098135,-0.015033,-0.041936
duration,0.041299,1.000000,-0.085348,-0.046998,0.025724,-0.028848,0.016672,-0.034745,-0.032329,-0.044218
campaign,-0.014169,-0.085348,1.000000,0.058742,-0.091490,0.176079,0.145021,0.007882,0.159435,0.161037
pdays,-0.043425,-0.046998,0.058742,1.000000,-0.587941,0.270684,0.058472,-0.092090,0.301478,0.381983
previous,0.050931,0.025724,-0.091490,-0.587941,1.000000,-0.415238,-0.164922,-0.051420,-0.458851,-0.514853
emp.var.rate,-0.019192,-0.028848,0.176079,0.270684,-0.415238,1.000000,0.755155,0.195022,0.970308,0.897173
cons.price.idx,-0.000482,0.016672,0.145021,0.058472,-0.164922,0.755155,1.000000,0.045835,0.657159,0.472560
cons.conf.idx,0.098135,-0.034745,0.007882,-0.092090,-0.051420,0.195022,0.045835,1.000000,0.276595,0.107054
euribor3m,-0.015033,-0.032329,0.159435,0.301478,-0.458851,0.970308,0.657159,0.276595,1.000000,0.942589
nr.employed,-0.041936,-0.044218,0.161037,0.381983,-0.514853,0.897173,0.472560,0.107054,0.942589,1.000000


#### Heatmap visualization

In [42]:
# Correlation matrix
corr_matrix = df[numeric_columns].corr()

# Create mask for upper triangle
mask = np.triu(
    np.ones_like(corr_matrix, dtype=bool)
)

# Apply mask
corr_matrix_masked = corr_matrix.mask(mask)

fig = px.imshow(
    corr_matrix_masked,
    text_auto=".2f",
    color_continuous_scale="BuGn",
    aspect="auto",
    title="Correlation Matrix Heatmap"
)

fig.update_layout(
    width=900,
    height=800
)

fig.show()

#### Strong Correlations

Several macroeconomic variables are highly correlated:

* **emp.var.rate ↔ euribor3m (0.97)**: Extremely strong positive correlation. Higher employment variation rates tend to occur alongside higher Euribor interest rates.
* **euribor3m ↔ nr.employed (0.94)**: Strong positive relationship between interest rates and employment levels.
* **emp.var.rate ↔ nr.employed (0.90)**: Employment variation and number of employees move closely together.
* **emp.var.rate ↔ cons.price.idx (0.76)**: Moderate-to-strong positive relationship between employment conditions and consumer prices.

These high correlations suggest **multicollinearity** among the economic indicators, meaning they provide overlapping information.

#### Moderate Correlations

* **pdays ↔ previous (-0.59)**: Clients contacted more frequently in previous campaigns tend to have fewer days since the last contact.
* **previous ↔ nr.employed (-0.51)**: Moderate negative relationship.
* **previous ↔ euribor3m (-0.46)** and **previous ↔ emp.var.rate (-0.42)**: Previous campaign contacts tend to decrease during periods associated with higher economic indicators.
* **pdays ↔ nr.employed (0.38)** and **pdays ↔ euribor3m (0.30)**: Weak-to-moderate positive relationships.

#### Weak or Negligible Correlations

Most client-level variables show little linear relationship with other features:

* **age** has very weak correlations with all variables (|r| < 0.10).
* **duration** is largely independent of other numerical features.
* **campaign** has only weak correlations with the remaining variables.
* **cons.conf.idx** shows limited correlation with most variables.

These features may still be useful predictors, but their relationships are likely non-linear or dependent on interactions with other variables.

#### Modeling Considerations

1. **Multicollinearity among economic variables**

   * `emp.var.rate`, `euribor3m`, and `nr.employed` are highly correlated.
   * Linear models may be affected by unstable coefficient estimates.
   * Consider feature selection, regularization, or dimensionality reduction if using linear methods.

2. **Behavioral variables are relatively independent**

   * Variables such as `duration`, `campaign`, and `age` provide information that is not strongly captured by other features.

3. **Correlation does not imply causation**

   * High correlations indicate variables move together, but do not establish cause-and-effect relationships.

#### Key Takeaway

The strongest relationships occur among the **macroeconomic indicators**, suggesting they describe similar economic conditions. In contrast, most **client and campaign features are weakly correlated**, indicating they contribute unique information and may provide additional predictive value for the classification task.

## Numerical Features vs Target Analysis

This analysis examines how numerical variables differ between clients who **subscribed to a term deposit (`y = yes`)** and those who **did not (`y = no`)**. The goal is to identify features that may help distinguish between the two groups.

In [44]:
df_plot = df.copy()

df_plot['target'] = df_plot['y'].apply(
    lambda x: 'Subscribed' if x == 'yes' else 'Not Subscribed'
)

features = [
    'age',
    'duration',
    'campaign',
    'pdays',
    'previous',
    'emp.var.rate',
    'cons.price.idx',
    'cons.conf.idx',
    'euribor3m',
    'nr.employed'
]

fig = make_subplots(
    rows=2,
    cols=5,
    subplot_titles=features
)

colors = {
    'Not Subscribed': '#4C78A8',
    'Subscribed': '#72B7B2'
}

positions = [
    (1,1), (1,2), (1,3), (1,4), (1,5),
    (2,1), (2,2), (2,3), (2,4), (2,5)
]

for feature, (r, c) in zip(features, positions):

    for target in ['Not Subscribed', 'Subscribed']:

        fig.add_trace(
            go.Histogram(
                x=df_plot.loc[df_plot['target'] == target, feature],
                name=target,
                marker_color=colors[target],
                opacity=0.6,
                showlegend=(feature == features[0])
            ),
            row=r,
            col=c
        )

    fig.update_xaxes(title_text=feature, row=r, col=c)
    fig.update_yaxes(title_text="Count", row=r, col=c)

fig.update_layout(
    title='Numerical Feature Distributions by Subscription Status',
    barmode='overlay',
    template='plotly_white',
    height=800,
    width=1800,
    legend_title='Subscription Status'
)

fig.show()

### Numerical Features vs Target — Observations

* **age**: The distributions largely overlap. Subscribers (`yes`) appear slightly more concentrated at older ages, suggesting older clients may be somewhat more likely to subscribe.

* **duration**: Subscribers tend to have longer call durations, while non-subscribers are more concentrated in shorter calls. This indicates a strong relationship between call engagement and subscription likelihood, although this feature introduces data leakage.

* **campaign**: Both groups are concentrated at low numbers of contacts. However, subscription rates appear to decrease as the number of campaign contacts increases.

* **pdays**: Most observations are concentrated at 999 (clients not previously contacted). Subscribers show a relatively higher proportion of lower `pdays` values, suggesting prior contact may increase the likelihood of subscription.

* **previous**: Most clients have zero previous contacts. Subscribers appear slightly more represented among clients with prior campaign interactions.

* **emp.var.rate**: Both groups follow a similar distribution pattern, although subscribers are consistently fewer across all values due to class imbalance. The distribution suggests some concentration of subscribers at lower employment variation rates.

* **cons.price.idx**: Subscribers and non-subscribers exhibit very similar distribution shapes. The lower counts for subscribers are primarily explained by the smaller number of positive cases rather than a clear effect of consumer prices.

* **cons.conf.idx**: Both groups follow a similar pattern across the range of values, suggesting consumer confidence alone has limited power to distinguish subscribers from non-subscribers. Subscriber counts remain lower throughout the distribution because of class imbalance.

* **euribor3m**: The overall shape is similar for both groups, although subscribers appear slightly more concentrated at lower interest rates. This may indicate some relationship between lower Euribor rates and subscription decisions.

* **nr.employed**: Both classes show similar distribution patterns, with subscribers consistently less frequent across all values. There is some indication that subscriptions occur more often during periods with lower employment levels.

### Key Takeaways

* **duration** appears to be the strongest differentiating feature, although it should be used cautiously due to leakage.
* **pdays** and **previous** show meaningful differences between subscribers and non-subscribers, highlighting the importance of previous campaign interactions.
* The macroeconomic variables show **substantial overlap between subscribers and non-subscribers**. While some differences can be observed for variables such as `euribor3m`, `emp.var.rate`, and `nr.employed`, the visual separation is modest. These features may still contribute predictive value when combined with other variables, but individually they do not appear to strongly distinguish between the two target classes.
* **age** shows a mild relationship with the target, while **cons.price.idx** and **cons.conf.idx** provide limited visual separation.

## Categorical Features vs Target Analysis

This analysis examines how the distribution of categorical variables differs between clients who **subscribed (`y = yes`)** and those who **did not subscribe (`y = no`)** to a term deposit. The objective is to identify categories associated with higher or lower subscription rates.

In [48]:
all_percentages = []

categorical_features = [
    'job',
    'marital',
    'education',
    'default',
    'housing',
    'loan',
    'contact',
    'month',
    'day_of_week',
    'poutcome'
]

for feature in categorical_features:

    tmp = (
        pd.crosstab(
            df_plot[feature],
            df_plot['target'],
            normalize='index'
        )
        .mul(100)
        .reset_index()
    )

    tmp.columns = [
        'category',
        'Not Subscribed (%)',
        'Subscribed (%)'
    ]

    tmp['feature'] = feature

    all_percentages.append(tmp)

percentages_df = pd.concat(all_percentages, ignore_index=True)

percentages_df = percentages_df[
    ['feature', 'category',
    'Not Subscribed (%)',
    'Subscribed (%)']
]

percentages_df

,feature,category,Not Subscribed (%),Subscribed (%)
0,job,admin.,86.857708,13.142292
1,job,blue-collar,93.099548,6.900452
2,job,entrepreneur,94.594595,5.405405
3,job,housemaid,90.000000,10.000000
4,job,management,90.740741,9.259259
5,job,retired,77.108434,22.891566
6,job,self-employed,91.823899,8.176101
7,job,services,91.094148,8.905852
8,job,student,76.829268,23.170732
9,job,technician,88.422576,11.577424


### 100% Stacked Bar Plots

In [46]:
df_plot = df.copy()

df_plot['target'] = df_plot['y'].apply(
    lambda x: 'Subscribed' if x == 'yes' else 'Not Subscribed'
)

categorical_features = [
    'job',
    'marital',
    'education',
    'default',
    'housing',
    'loan',
    'contact',
    'month',
    'day_of_week',
    'poutcome'
]

fig = make_subplots(
    rows=2,
    cols=5,
    subplot_titles=categorical_features
)

colors = {
    'Not Subscribed': '#4C78A8',
    'Subscribed': '#72B7B2'
}

positions = [
    (1,1), (1,2), (1,3), (1,4), (1,5),
    (2,1), (2,2), (2,3), (2,4), (2,5)
]

for feature, (r, c) in zip(categorical_features, positions):

    tmp = pd.crosstab(
        df_plot[feature],
        df_plot['target'],
        normalize='index'
    ) * 100

    for target in ['Not Subscribed', 'Subscribed']:

        if target in tmp.columns:
            fig.add_trace(
                go.Bar(
                    x=tmp.index.astype(str),
                    y=tmp[target],
                    name=target,
                    marker_color=colors[target],
                    showlegend=(feature == categorical_features[0])
                ),
                row=r,
                col=c
            )

    fig.update_xaxes(title_text=feature, row=r, col=c, tickangle=45)
    fig.update_yaxes(title_text="Percentage (%)", row=r, col=c)

fig.update_layout(
    title='Categorical Features vs Subscription Status (100% Stacked)',
    barmode='stack',
    template='plotly_white',
    height=900,
    width=1800,
    legend_title='Subscription Status'
)

fig.show()

##### Client Characteristics

##### Job

* Subscription rates vary considerably across occupations.
* **Students (23.2%)** and **retired clients (22.9%)** show the highest subscription rates.
* **Unemployed clients (17.1%)** also exhibit above-average subscription rates.
* **Blue-collar (6.9%)** and **entrepreneur (5.4%)** clients show the lowest subscription rates.
* This suggests occupation is a potentially important predictor of subscription behavior.

#### Marital Status

* **Single clients (13.4%)** have a higher subscription rate than married (10.0%) and divorced (9.6%) clients.
* Differences exist but are relatively modest.

#### Education

* Higher subscription rates are observed among clients with **unknown education (15.6%)**, **university degrees (13.1%)**, and **professional courses (12.1%)**.
* Clients with basic education levels generally show lower subscription rates.
* The single illiterate observation is not representative and should not be interpreted.

#### Financial Status

#### Default

* Clients with **no default history** have a subscription rate of **12.1%**.
* Clients labeled as **unknown** have a substantially lower rate (**6.1%**).
* The single client with default status "yes" did not subscribe, but the sample size is too small for meaningful conclusions.

#### Housing Loan

* Subscription rates are very similar between clients with and without housing loans (~11%).
* This variable appears to have limited individual predictive power.

#### Personal Loan

* Similar subscription rates are observed across categories, suggesting a weak relationship with the target.

#### Contact Information

#### Contact Type

* Clients contacted through **cellular phones (14.1%)** are substantially more likely to subscribe than those contacted via **telephone (5.2%)**.
* This is one of the strongest categorical effects observed in the dataset.

#### Campaign Timing

#### Month

* Strong seasonal effects are evident.
* The highest subscription rates occur in:

  * **March (58.3%)**
  * **December (54.5%)**
  * **September (40.6%)**
  * **October (36.2%)**
* The lowest subscription rate occurs in **May (6.5%)**, despite being the most common contact month.
* Some months have relatively small sample sizes, so extreme percentages should be interpreted cautiously.

#### Day of Week

* Subscription rates are nearly identical across weekdays (approximately 10–11%).
* This variable appears to have little influence on campaign success.

#### Previous Campaign Information

#### Previous Campaign Outcome (`poutcome`)

* This is the strongest categorical predictor in the dataset.
* Clients whose previous campaign was a **success** have a subscription rate of **64.8%**.
* Clients with a previous **failure** have a moderate subscription rate (**14.8%**).
* Clients with **no previous campaign history** (`nonexistent`) have the lowest rate (**8.3%**).

This indicates that previous campaign success is strongly associated with future subscription behavior.

#### Key Takeaways

The most informative categorical features appear to be:

1. **poutcome** (previous campaign outcome)
2. **month** (seasonality effects)
3. **contact** (cellular vs telephone)
4. **job** (especially retired and student categories)
5. **education**

In contrast, **housing**, **loan**, and **day_of_week** show little variation in subscription rates and may have weaker standalone predictive power.

---

## Potential Target Variable

**Candidate target:**

* `y` — Indicates whether the client subscribed to a term deposit (`yes` or `no`).

**Potential ML task:**

* **Binary Classification**

### Rationale

The objective of this dataset is to predict whether a client will subscribe to a bank term deposit based on demographic characteristics, financial information, previous marketing interactions, and economic indicators.

The target variable `y` contains two possible outcomes:

* **yes**: The client subscribed to a term deposit.
* **no**: The client did not subscribe.

Since the target consists of two mutually exclusive categories, the problem is naturally formulated as a **binary classification task**.

A key characteristic of the target variable is its **class imbalance**:

* **No:** 3,668 observations (89.05%)
* **Yes:** 451 observations (10.95%)

This imbalance should be considered during model training and evaluation, as standard accuracy may provide misleading results. Metrics such as **Precision**, **Recall**, **F1-Score**, and **ROC-AUC** are more appropriate for assessing model performance.

The target variable is also highly relevant from a business perspective, as accurately identifying clients likely to subscribe can help the bank improve marketing efficiency, reduce campaign costs, and increase conversion rates.

---

# Planned Feature Preprocessing

Before model training, the dataset will undergo preprocessing to ensure variables are in a suitable format for machine learning algorithms and to minimize the impact of skewness, outliers, and hidden missing values.

### Planned Steps

* Remove non-informative variables when appropriate.
* Convert the target variable (`y`) into a binary label using **LabelEncoder** (`no=0`, `yes=1`).
* Handle hidden missing values represented by `"unknown"` before encoding.
* Apply **RobustScaler** to highly skewed numerical variables with significant outliers.
* Apply **StandardScaler** to numerical variables with approximately symmetric distributions and no major outlier issues.
* Apply **MinMaxScaler** to non-normal numerical variables without significant outliers.
* Encode nominal categorical variables using **One-Hot Encoding (`drop='first'`)**.
* Encode ordinal categorical variables using **OrdinalEncoder**.

| Column           | Variable Type       | Planned Treatment                                      | Encoding / Scaling Strategy       | Reason                                                   |
| ---------------- | ------------------- | ------------------------------------------------------ | --------------------------------- | -------------------------------------------------------- |
| `age`            | Numerical           | Keep                                                   | RobustScaler                  | Contains outliers and mild skewness                      |
| `job`            | Nominal Categorical | Keep                                                   | One-Hot Encoding (`drop='first'`) | No natural order                                         |
| `marital`        | Nominal Categorical | Keep                                                   | One-Hot Encoding (`drop='first'`) | No natural order                                         |
| `education`      | Ordinal Categorical | Keep                                                   | OrdinalEncoder                    | Natural educational progression                          |
| `default`        | Nominal Categorical | Handle `unknown` values                                | One-Hot Encoding (`drop='first'`) | No natural order                                         |
| `housing`        | Nominal Categorical | Handle `unknown` values                                | One-Hot Encoding (`drop='first'`) | Binary categorical variable                              |
| `loan`           | Nominal Categorical | Handle `unknown` values                                | One-Hot Encoding (`drop='first'`) | Binary categorical variable                              |
| `contact`        | Nominal Categorical | Keep                                                   | One-Hot Encoding (`drop='first'`) | No natural order                                         |
| `month`          | Ordinal Categorical | Keep                                                   | OrdinalEncoder                    | Temporal ordering                                        |
| `day_of_week`    | Ordinal Categorical | Keep                                                   | OrdinalEncoder                    | Temporal ordering                                        |
| `duration`       | Numerical           | Keep for benchmark models; remove for realistic models | RobustScaler                  | Highly skewed with significant outliers and leakage risk |
| `campaign`       | Numerical           | Keep                                                   | RobustScaler                  | Strong right skew and outliers                           |
| `pdays`          | Numerical           | Transform special value (999) before modeling          | RobustScaler                  | Highly skewed and contains outliers                      |
| `previous`       | Numerical           | Keep                                                   | RobustScaler                  | Highest proportion of outliers                           |
| `poutcome`       | Nominal Categorical | Keep                                                   | One-Hot Encoding (`drop='first'`) | No natural order                                         |
| `emp.var.rate`   | Numerical           | Keep                                                   | MinMaxScaler                      | No outliers detected; non-normal distribution            |
| `cons.price.idx` | Numerical           | Keep                                                   | MinMaxScaler                      | No outliers detected; non-normal distribution            |
| `cons.conf.idx`  | Numerical           | Keep                                                   | RobustScaler                  | Outliers detected and non-normal distribution            |
| `euribor3m`      | Numerical           | Keep                                                   | MinMaxScaler                      | No outliers detected; non-normal distribution            |
| `nr.employed`    | Numerical           | Keep                                                   | MinMaxScaler                      | No outliers detected; non-normal distribution            |
| `y`              | Target Variable     | Convert to binary                                      | LabelEncoder                      | Required for binary classification                       |


### Additional Notes

* The value `"unknown"` appears in several categorical variables and represents hidden missing data. Different strategies (keeping as a category, imputation, or removal) will be evaluated during preprocessing.
* The variable `duration` is known only after a call has ended and may cause **data leakage**. Two modeling approaches may be considered:

  * **Benchmark model:** includes `duration`.
  * **Real-world predictive model:** excludes `duration`.
* The variable `pdays` contains a special value (`999`) that does not represent a true number of days and should be treated separately before scaling.

---

# Conclusions and Insights

## Dataset Overview

The dataset contains **4,119 client records** and **20 input features**, describing client demographics, financial status, marketing campaign interactions, and macroeconomic conditions. The objective is to predict whether a client subscribes to a term deposit (`y`), making this a **binary classification problem**.

No duplicated records or true missing values were found. However, several categorical variables contain the value **`unknown`**, which represents hidden missing information and should be handled during preprocessing.

## Data Quality Assessment

### Strengths

* No duplicated observations.
* No null values in any column.
* Consistent data types across all features.
* Sufficient sample size for classification modeling.

### Potential Issues

* Hidden missing values encoded as `unknown`.
* Special value `999` in `pdays` represents "not previously contacted" rather than an actual number of days.
* `duration` introduces potential data leakage because it is only known after the call has ended.
* Several numerical variables exhibit strong skewness and outliers.

## Target Variable Insights

The target variable is highly imbalanced:

* **No subscription:** 3,668 observations (89.05%)
* **Subscription:** 451 observations (10.95%)

This imbalance should be considered during model evaluation, as accuracy alone may provide misleading results. Metrics such as Precision, Recall, F1-Score, and ROC-AUC will be more informative.

## Numerical Feature Insights

### Client Characteristics

* `age` shows a slight tendency for subscribers to be older than non-subscribers, although the distributions largely overlap.

### Campaign Interaction Variables

* `duration` appears to be one of the strongest predictors, with subscribers generally associated with longer call durations.
* `campaign` suggests diminishing returns from repeated contacts, as higher numbers of contacts are associated with lower subscription rates.
* `pdays` and `previous` indicate that prior campaign interactions may influence subscription decisions.
* Most clients have never been contacted before (`pdays = 999`, `previous = 0`).

### Macroeconomic Variables

* `emp.var.rate`, `euribor3m`, and `nr.employed` show noticeable differences between subscribers and non-subscribers.
* `cons.price.idx` and `cons.conf.idx` exhibit substantial overlap between classes and appear less informative individually.
* Macroeconomic conditions may contribute predictive value when combined with other features.

## Categorical Feature Insights

### Most Informative Variables

#### Previous Campaign Outcome (`poutcome`)

This appears to be the strongest categorical predictor:

* Success: 64.8% subscription rate
* Failure: 14.8% subscription rate
* Nonexistent: 8.3% subscription rate

Clients who previously responded positively are much more likely to subscribe again.

#### Contact Method (`contact`)

* Cellular: 14.1% subscription rate
* Telephone: 5.2% subscription rate

Clients contacted via mobile phones are substantially more likely to subscribe.

#### Job (`job`)

Highest subscription rates:

* Student: 23.2%
* Retired: 22.9%
* Unemployed: 17.1%

Lowest subscription rates:

* Entrepreneur: 5.4%
* Blue-collar: 6.9%

Occupation appears to influence subscription behavior.

#### Month (`month`)

Strong seasonality is present:

* Highest subscription rates: March, December, September, and October
* Lowest subscription rate: May

Campaign timing appears to affect success rates.

### Less Informative Variables

* `housing`
* `loan`
* `day_of_week`

These features show similar subscription rates across categories and appear to have limited standalone predictive power.

## Correlation Analysis

Most client and campaign variables show weak linear relationships with each other, indicating they may contribute unique information.

The strongest correlations are found among the macroeconomic indicators:

| Variables                  | Correlation |
| -------------------------- | ----------: |
| emp.var.rate ↔ euribor3m   |        0.97 |
| euribor3m ↔ nr.employed    |        0.94 |
| emp.var.rate ↔ nr.employed |        0.90 |

These strong relationships suggest **multicollinearity**, which should be considered when using linear models.

## Outlier Analysis

Outliers are concentrated in behavioral variables:

| Feature       | Outlier % |
| ------------- | --------: |
| previous      |    14.47% |
| duration      |     7.06% |
| campaign      |     5.71% |
| pdays         |     3.88% |
| cons.conf.idx |     1.04% |
| age           |     0.95% |

Most outliers appear to represent genuine customer behavior rather than data errors. Therefore, they should be handled through robust scaling rather than removal.

## Key Predictive Signals Identified

Based on the exploratory analysis, the features with the strongest potential predictive value are:

1. `poutcome`
2. `duration` *(for benchmark models only)*
3. `contact`
4. `month`
5. `job`
6. `pdays`
7. `previous`
8. `euribor3m`
9. `emp.var.rate`

These variables show the clearest differences between subscribers and non-subscribers.

## Recommendations for Modeling

* Convert `y` into a binary target using LabelEncoder.
* Handle `unknown` values explicitly before encoding.
* Treat `999` in `pdays` as a special category or engineered feature.
* Apply **RobustScaler** to variables with detected outliers:

  * `age`
  * `duration`
  * `campaign`
  * `pdays`
  * `previous`
  * `cons.conf.idx`
* Apply **MinMaxScaler** to the remaining numerical variables.
* Use **One-Hot Encoding** for nominal categorical variables.
* Use **Ordinal Encoding** for ordinal variables (`education`, `month`, `day_of_week`).
* Use evaluation metrics appropriate for imbalanced classification problems.

## Final Takeaway

The dataset is generally clean and suitable for predictive modeling. The strongest signals come from **previous campaign outcomes, contact strategy, campaign timing, customer occupation, and prior marketing interactions**. While macroeconomic indicators show meaningful patterns, their high correlation suggests they capture similar economic conditions. Careful handling of class imbalance, hidden missing values, outliers, and potential data leakage will be essential for building a reliable and realistic subscription prediction model.

---

# Appendix

## References

- [Bank Marketing Dataset from the UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing)
- Machine Learning notebooks from the bootcamp repository (*sonda2026* branch)
- Machine Learning notebooks from the shared Assistanship Google Drive folder
- [Pandas Documentation](https://pandas.pydata.org/docs/user_guide/index.html?utm_source=chatgpt.com)
- [Plotly Documentation](https://plotly.com/python/?utm_source=chatgpt.com)

## Acknowledgements

I would like to thank my instructors for their guidance, continuous support, and encouragement throughout the development of this project.

I also acknowledge the use of AI-assisted tools to support debugging, code review, documentation, and the exploration of machine learning concepts during the analysis process. All modeling decisions, interpretations, and conclusions presented in this work were made independently and remain my own responsibility.

---

### Tools and Technologies

- Python
- Pandas
- NumPy
- SciPy
- Plotly
- Jupyter Notebook